# stage3d_control
130mm / historical v5 labels / seed42. Auxiliary ranking lambda=0.0.
Mount competition, rsna-dinov2-weights and rsna-knee-v5-labels. The small trust asset is embedded.
Rule-selected candidates are not clinical ground truth. Gold is a reused development set.


In [ ]:
# ============================================================
# v4: Environment Setup
# ============================================================
# ★ 竞赛环境已预装 timm / pydicom / opencv / sklearn，无需 pip install
# 如果缺少包，将其打包为 Kaggle Dataset 挂载即可
print('Setup complete.')


In [ ]:
# ============================================================
# v4: Imports
# ============================================================
from __future__ import annotations
import gc, math, os, re, sys, time
from pathlib import Path
from collections import defaultdict
from concurrent.futures import ThreadPoolExecutor

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import timm
import pydicom
import cv2
from sklearn.metrics import roc_auc_score

IS_MAIN = True
print('Imports OK.')


In [ ]:
# ============================================================
# v5: Configuration — 288px/130mm 奈奎斯特分辨率 + v5 融合软标签 (teacher-student)
# ============================================================

TARGET_COLUMNS = [
    'ACL', 'MCL', 'Medial Meniscus', 'Lateral Meniscus',
    'Medial OA', 'Lateral OA', 'PF OA',
    'Effusion', 'Synovitis', "Baker's",
    'Contusion', 'Fracture',
]
N_CLASSES = len(TARGET_COLUMNS)

# Soft-label columns
PROB_COLS   = [f'prob_{c}' for c in TARGET_COLUMNS]
WEIGHT_COLS = [f'weight_{c}' for c in TARGET_COLUMNS]
MASK_COLS   = [f'mask_{c}' for c in TARGET_COLUMNS]

# ---- 6 Clinical Slots ----
SLOTS = [
    ("SAG_FLUID_FS",   "Sagittal", True,  True),
    ("COR_FLUID_FS",   "Coronal",  True,  True),
    ("AX_FLUID_FS",    "Axial",    True,  True),
    ("SAG_FLUID_NOFS", "Sagittal", True,  False),
    ("COR_T1",         "Coronal",  False, False),
    ("SAG_T1",         "Sagittal", False, False),
]
N_SLOT = len(SLOTS)

# ---- Anatomical Priors ----
SLOT_PRIORS = {
    "ACL": (0, 3, 5), "MCL": (1, 4),
    "Medial Meniscus": (0, 1, 3, 4), "Lateral Meniscus": (0, 1, 3, 4),
    "Medial OA": (1, 4, 5), "Lateral OA": (1, 4, 5),
    "PF OA": (0, 2, 5), "Effusion": (0, 2), "Synovitis": (0, 2),
    "Baker's": (0,), "Contusion": (0, 1, 2), "Fracture": (0, 1, 2, 4, 5),
}

# ---- Diagnostic-specific TTA pooling ----
# 局部病灶用 max（保留最强信号），ACL/MCL 用 top2，弥漫性病变用 mean
# 与 0.91 notebook 的 TTA_TARGET_POOL 逐项一致
DIAG_POOL = {
    "Fracture": "max", "Contusion": "max",
    "Medial Meniscus": "max", "Lateral Meniscus": "max",
    "Baker's": "max",
    "ACL": "top2", "MCL": "top2",
    # ★ 0.91 同款: 仅用无 jitter 原始视图平均
    #   (jitter TTA 开启时生效; 关闭时所有视图皆原始, 等价于 mean)
    "Synovitis": "original_mean",
    # 其余（OA, Effusion）默认 mean
}

# ---- Jitter TTA 增广 (0.91 notebook augment() 移植) ----
AUG_ROT_DEG = 8.0          # 旋转 ±8°
AUG_SCALE = 0.08           # 缩放 +[0, 8%]
AUG_SHIFT = 0.05           # 平移 ±5%
AUG_INTENSITY = 0.1        # 强度 ±10%
AUG_SEED = 42              # 增广视图固定种子（确定性, 跨验证/测试/提交可复现）

CFG = {
    # --- Paths ---
    'comp_input':   '/kaggle/input/competitions/rsna-knee-abnormality-detection',
    # ★ v5 融合标签数据集 (本地 scripts/build_v5_labels.py 生成 v5_labels.csv 后上传)
    'label_input':  '/kaggle/input/datasets/easoncyy/rsna-knee-v5-labels',
    'dicom_subdir': 'train_series',
    'output_dir':   '/kaggle/working',

    # --- Data ---
    'image_size': 288,             # ★ v5: 288px@130mm = 0.451mm/px 满足奈奎斯特
                                   #   (v4: 224px@160mm = 0.714mm/px 不满足)
                                   #   RAM 缓存 ~19.7GB + 运行时 ~4GB → 需要 T4x2 (~29GB 系统内存);
                                   #   P100 (16GB RAM) 请改用 256 + cache_slices 7 (12.1GB 缓存, 0.508mm/px)
    'crop_mm': 130.0,              # ★ v5: 130mm FOV (膝关节 ~130mm, 无浪费像素)
    'cache_slices': 9,
    'group_size': 3,               # 3 adjacent slices → RGB channels
    'center_pct': (0.2, 0.8),

    # --- Model ---
    'dinov2_variant': 'vit_small_patch14_dinov2.lvd142m',
    'dinov2_weights': '/kaggle/input/rsna-dinov2-weights/dinov2_vits14.pth',  # ★ 竞赛禁网，权重打包为 Dataset
    'cls_dim': 384,
    'feature_dim': 1152,
    'slot_hidden': 256,
    'num_classes': 12,
    'unfreeze_layers': 6,
    'dropout': 0.2,

    # --- Training ---
    'batch_size': 6,
    'grad_accum_steps': 2,
    'epochs': 30,                  # ★ v5: 40→30 (288px 计算量 1.65×, 靠标签质量补偿)
    'seed': 42,                    # ★ seed 自集成: 每换一个 seed 跑一个会话 (42/142/242 → v5s1/s2/s3)
    'lr': 2e-4,
    'backbone_lr': 1e-5,
    'weight_decay': 1e-4,
    'lr_t0': 15,
    'lr_t_mult': 2,
    'lr_eta_min': 1e-6,
    'grad_clip': 1.0,
    'early_stop_patience': 12,
    'mixed_precision': True,
    'num_workers': 2,

    # --- ★ v5 ---
    'ema_decay': 0.999,            # EMA 权重平均 (v4 沿用)
    'max_train_minutes': 420,      # ★ 训练墙钟保护 (Kaggle 9h 会话上限内留出推理时间)
    'diag_pool_train': True,       # 训练时也做诊断池化
    'tta_jitter': True,            # ★ jitter TTA (0.91 移植): 每窗口额外 1 个确定性增广视图,
                                   #   视图平均后再窗口池化; 验证/推理成本 ×2 (训练不变)
    'hdr_threads': 8,              # DICOM 并行读取线程数
    'pix_threads': 4,              # 像素解码并行线程数
}

# ---- 全局随机种子 — seed 自集成的成员独立性来源 ----
#   换 CFG['seed'] = 新成员: 训练顺序/头初始化/优化路径全部不同 → 半独立
import random
random.seed(CFG['seed'])
np.random.seed(CFG['seed'])
torch.manual_seed(CFG['seed'])
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(CFG['seed'])

SEED_TAG = f's{CFG["seed"]}'              # 产物文件名后缀 (s42/s142/s242)
CKPT_NAME = f'best_model_{SEED_TAG}.pt'   # 15 保存 / 17 加载共用

# Device
N_GPUS = torch.cuda.device_count()
DEVICE = torch.device('cuda' if N_GPUS > 0 else 'cpu')

if IS_MAIN:
    print(f'GPUs: {N_GPUS} | Device: {DEVICE}')
    print(f'--- v5: 288px/130mm + Fused Soft Labels (text×OOF teacher) + Confidence-weighted BCE ---')
    for k, v in CFG.items():
        print(f'  {k}: {v}')

CFG.update(experiment_name='stage3d_control', rank_lambda=0.0, rank_margin=0.1, rank_min_cases=20, trust_input='/kaggle/working/stage3d_embedded')

# The 122-KB immutable eligibility table is embedded to remove a fragile Kaggle mount.
import base64 as _stage3d_base64
import hashlib
_embedded_dir = Path(CFG['trust_input'])
_embedded_dir.mkdir(parents=True, exist_ok=True)
_embedded_bytes = _stage3d_base64.b64decode('...')
assert hashlib.sha256(_embedded_bytes).hexdigest() == '92600b0df5bee567502240533fd8e47281225ca3a55f429b5e57cd74fc6515f6'
(_embedded_dir / 'stage3d_trust.npz').write_bytes(_embedded_bytes)

# Resolve documented old/new Kaggle mount layouts; missing weights must stop.
def resolve_asset(config_key, filename, dataset_slug):
    configured = Path(CFG[config_key])
    candidates = [configured / filename if config_key != 'dinov2_weights' else configured,
                  Path('/kaggle/input') / dataset_slug / filename,
                  Path('/kaggle/input/datasets/easoncyy') / dataset_slug / filename]
    found = next((p for p in candidates if p.is_file()), None)
    if found is None:
        raise FileNotFoundError(f'Mount {dataset_slug}/{filename}; checked {candidates}')
    CFG[config_key] = str(found if config_key == 'dinov2_weights' else found.parent)
resolve_asset('label_input', 'v5_labels.csv', 'rsna-knee-v5-labels')
resolve_asset('dinov2_weights', 'dinov2_vits14.pth', 'rsna-dinov2-weights')
print('Stage 3D configuration:', CFG)


In [ ]:
# ============================================================
# v4: Slot Matching + Laterality Detection + DICOM Header Annotation
# ============================================================

# ---- DICOM Header Annotation (Ref1: annotate_sequences) ----
_SEP = re.compile(r'[_\-.]')
_FATSAT_RX = re.compile(
    r'\bfs\b|fatsat|fat sat|\bstir\b|\bspair\b|\bspir\b|\bwe\b|'
    r'water excit|\btirm\b|\bsting\b|\bfatsup\b'
)
_T1_RX = re.compile(r'\bt1\b|\bt1w\b')
_T2_RX = re.compile(r'\bt2\b|\bt2w\b')
_PD_RX = re.compile(r'\bpd\b|\bpdw\b|proton|\bdp\b|dens')

FATSAT_OPTS = {'FS', 'FATSAT', 'FAT_SAT', 'FSAT'}

_HDR_TAGS = [
    'SeriesDescription', 'SequenceName', 'ScanOptions', 'ScanningSequence',
    'RepetitionTime', 'EchoTime', 'Laterality', 'ImageLaterality',
    'ImagePositionPatient', 'PixelSpacing',
]


def _tag_side(group):
    """从 DICOM Laterality 标签推断侧性。"""
    values = [str(x).strip().upper() for x in group.get('Laterality', pd.Series(dtype=object)).dropna()]
    if 'ImageLaterality' in group.columns:
        values += [str(x).strip().upper() for x in group['ImageLaterality'].dropna()]
    values = [x[0] for x in values if x and x[0] in ('L', 'R')]
    return values[0] if values else None


def _position_side(group, min_offset_mm=5.0):
    """从 ImagePositionPatient[0] 推断侧性：DICOM LPS 中 +x = 患者左侧。"""
    xs = []
    for raw in group.get('ImagePositionPatient', pd.Series(dtype=object)).dropna():
        try:
            xs.append(float(str(raw).split('|')[0]))
        except Exception:
            pass
    if not xs:
        return None
    median_x = float(np.median(xs))
    if abs(median_x) < min_offset_mm:
        return None
    return 'R' if median_x < 0 else 'L'


def detect_laterality(headers_df):
    """为每个 study 确定侧性（左/右），结合标签和几何位置。"""
    tagged, positioned = {}, {}
    for study_uid, group in headers_df.groupby('StudyInstanceUID'):
        tagged[study_uid] = _tag_side(group)
        positioned[study_uid] = _position_side(group)

    comparable = [s for s in tagged if tagged[s] and positioned[s]]
    agreement = float(np.mean([
        tagged[s] == positioned[s] for s in comparable
    ])) if comparable else np.nan

    use_position = bool(comparable) and np.isfinite(agreement) and agreement >= 0.85

    resolved = {
        uid: (tagged[uid] or (positioned[uid] if use_position else None))
        for uid in tagged
    }
    coverage = float(np.mean([v is not None for v in resolved.values()]))

    if IS_MAIN:
        print(f'Laterality: tag_coverage={len([v for v in tagged.values() if v])/max(len(tagged),1):.1%}, '
              f'agreement={agreement:.1%} on {len(comparable)} studies, '
              f'final_coverage={coverage:.1%}')
    return resolved


def annotate_sequences(df):
    """从 DICOM header 推断 Fluid/FatSat/Weight，作为 train_series.csv 的 fallback。"""
    df = df.copy()

    # Fat suppression detection
    desc = (df.get('SeriesDescription', '').fillna('') + ' ' +
            df.get('SequenceName', '').fillna(''))
    desc = desc.str.lower().str.replace(_SEP, ' ', regex=True)

    scan_options = df.get('ScanOptions', '').fillna('').str.upper().str.split('|')
    option_fatsat = scan_options.apply(
        lambda tokens: any(t.strip() in FATSAT_OPTS for t in tokens))
    df['fatsat_detected'] = desc.str.contains(_FATSAT_RX) | option_fatsat

    # Weight detection
    tr = pd.to_numeric(df.get('RepetitionTime', np.nan), errors='coerce')
    te = pd.to_numeric(df.get('EchoTime', np.nan), errors='coerce')
    named_t1 = desc.str.contains(_T1_RX)
    named_t2 = desc.str.contains(_T2_RX)
    named_pd = desc.str.contains(_PD_RX)

    df['weight'] = np.where(
        named_t1 & ~named_t2 & ~named_pd, 'T1',
        np.where(named_t2 & ~named_pd, 'T2',
                 np.where(named_pd, 'PD',
                          np.where(tr < 800, 'T1',
                                   np.where(te > 60, 'T2',
                                            np.where(tr >= 800, 'PD', 'UNK'))))))
    df['fluid_detected'] = df['weight'].isin(['PD', 'T2'])

    return df


# ---- Slot Matching ----
def match_slots_for_study(study_series_df):
    """为单个 study 的每个 slot 匹配最优 series。"""
    slots_found = {}
    for slot_name, plane, fluid, fatsat in SLOTS:
        candidates = study_series_df[
            (study_series_df['Anatomical_Plane'] == plane)
            & (study_series_df['Fluid_Sensitive'] == (1 if fluid else 0))
            & (study_series_df['Fat_Suppression'] == (1 if fatsat else 0))
        ]
        if len(candidates) == 0 and not fluid:
            candidates = study_series_df[
                (study_series_df['Anatomical_Plane'] == plane)
                & (study_series_df['Fluid_Sensitive'] == 0)
            ]
        if len(candidates) > 0:
            best = candidates.sort_values('n_slices', ascending=False).iloc[0]
            slots_found[slot_name] = {
                'series_uid': best['SeriesInstanceUID'],
                'dir': best['dir'],
                'n_slices': int(best['n_slices']),
                'plane': plane,
            }
        else:
            slots_found[slot_name] = None
    return slots_found


def build_study_slot_map(series_meta, dicom_root):
    """为所有 study 构建 slot→series 映射。"""
    df = series_meta.copy()
    df['StudyInstanceUID'] = df['StudyInstanceUID'].astype(str)
    df['SeriesInstanceUID'] = df['SeriesInstanceUID'].astype(str)

    # 计算 DICOM 目录和切片数
    dirs, n_slices_list = [], []
    for _, row in df.iterrows():
        d = str(dicom_root / row['StudyInstanceUID'] / row['SeriesInstanceUID'])
        dirs.append(d)
        if os.path.isdir(d):
            files = [f for f in os.listdir(d) if os.path.isfile(os.path.join(d, f))]
            n_dcm = len([f for f in files if f.endswith('.dcm')])
            if n_dcm == 0:
                n_dcm = len([f for f in files if not f.startswith('.')])
            n_slices_list.append(n_dcm)
        else:
            n_slices_list.append(0)
    df['dir'] = dirs
    df['n_slices'] = n_slices_list

    slot_map, study_series_map = {}, {}
    for study_uid, grp in df.groupby('StudyInstanceUID'):
        study_series_map[study_uid] = grp
        slot_map[study_uid] = match_slots_for_study(grp)

    # 统计
    slot_counts = {}
    for slots in slot_map.values():
        for name, sid in slots.items():
            slot_counts[name] = slot_counts.get(name, 0) + (1 if sid is not None else 0)

    if IS_MAIN:
        n_studies = len(slot_map)
        print(f'Slot map: {n_studies} studies')
        for name, count in slot_counts.items():
            print(f'  {name:<18s}: {count:5d}/{n_studies} ({count/n_studies*100:.0f}%)')

    return slot_map, study_series_map

print('Slot matching v4 ready.')


In [ ]:
# ============================================================
# v4: DICOM I/O — 空间排序 + 物理裁剪 + 侧性归一化 + 并行读取
# ============================================================

# ---- 空间切片排序 (Ref2: dominant_axis) ----
PLANE_AXIS = {"Sagittal": 0, "Coronal": 1, "Axial": 2}

def _list_dcm_files(series_dir):
    """列出 DICOM 文件（不依赖 .dcm 扩展名，竞赛 test 集无后缀）。"""
    sd = Path(series_dir)
    if not sd.is_dir():
        return []
    all_files = sorted(f.name for f in sd.iterdir() if f.is_file())
    dcm = [f for f in all_files if f.endswith('.dcm')]
    return dcm if dcm else [f for f in all_files if not f.startswith('.')]

def spatially_sorted_files(series_dir, plane=None):
    """按 ImagePositionPatient 在切片法线方向上的投影排序。
    文件名排序的 Spearman 相关系数仅 0.009——完全随机。
    """
    series_dir = Path(series_dir)
    files = _list_dcm_files(series_dir)
    if not files:
        return []

    axis = PLANE_AXIS.get(plane, 2)
    rows = []
    for fname in files:
        try:
            ds = pydicom.dcmread(
                str(series_dir / fname), stop_before_pixels=True, force=True,
                specific_tags=['ImagePositionPatient', 'InstanceNumber'])
            ipp = getattr(ds, 'ImagePositionPatient', None)
            instance = getattr(ds, 'InstanceNumber', None)
            if ipp is not None and len(ipp) >= 3:
                candidate = np.array(ipp[:3], dtype=np.float64)
                pos = float(candidate[axis]) if np.isfinite(candidate).all() else None
            else:
                pos = None
            inst_val = float(instance) if instance is not None else None
        except Exception:
            pos, inst_val = None, None
        rows.append((fname, pos, inst_val))

    positioned = [r for r in rows if r[1] is not None]
    threshold = max(2, int(0.8 * len(rows)))

    if len(positioned) >= threshold:
        # 主排序：通过平面坐标
        rows.sort(key=lambda r: (
            r[1] if r[1] is not None else 0.0,
            r[2] if r[2] is not None else float('inf'),
        ))
    elif sum(r[2] is not None for r in rows) >= threshold:
        rows.sort(key=lambda r: (
            r[2] if r[2] is not None else float('inf'),
        ))
    # else: 保持文件名顺序

    return [r[0] for r in rows]


# ---- 侧性归一化 ----
def normalise_laterality(image, plane, laterality):
    """右膝映射为左膝：冠/轴面水平翻转，矢面反转切片顺序。"""
    if laterality != 'R':
        return image
    # image: [N_slices, H, W] numpy
    if plane in ('Coronal', 'Axial'):
        return np.flip(image, axis=-1).copy()  # 水平翻转
    else:
        return np.flip(image, axis=0).copy()    # 反转切片顺序


# ---- 物理裁剪 ----
def physical_crop(volume, px, crop_mm=160.0):
    """基于 PixelSpacing 裁剪到固定物理 FOV，消除不同扫描仪的空间尺度差异。"""
    if px is None or not np.isfinite(px) or px <= 0:
        return volume
    desired = int(round(crop_mm / px))
    h, w = volume.shape[1], volume.shape[2]
    if not (16 < desired < min(h, w)):
        return volume
    cy, cx = h // 2, w // 2
    half = desired // 2
    return volume[:, max(0, cy - half):cy + half, max(0, cx - half):cx + half]


# ---- 读取单 series 为 volume ----
def read_series_volume(series_dir, plane=None, laterality=None,
                       image_size=224, crop_mm=160.0):
    """读取 DICOM 序列 → 空间排序 → 物理裁剪 → 侧性归一化 → 归一化 → 缩放。"""
    sorted_files = spatially_sorted_files(series_dir, plane)
    if not sorted_files:
        return None, None

    series_dir = Path(series_dir)
    slices_info = []
    px = None

    for fname in sorted_files:
        try:
            ds = pydicom.dcmread(str(series_dir / fname), force=True)
            img = ds.pixel_array.astype(np.float32)

            # Rescale
            slope = float(getattr(ds, 'RescaleSlope', 1) or 1)
            intercept = float(getattr(ds, 'RescaleIntercept', 0) or 0)
            img = img * slope + intercept

            # PixelSpacing (取第一个有效值)
            if px is None:
                ps = getattr(ds, 'PixelSpacing', None)
                if ps is not None and len(ps) >= 1:
                    try:
                        px = float(ps[0])
                    except Exception:
                        pass

            slices_info.append(img)
        except Exception:
            slices_info.append(np.zeros((image_size, image_size), dtype=np.float32))

    if not slices_info:
        return None, None

    volume = np.stack(slices_info, axis=0)  # [N, H, W]

    # 物理裁剪
    volume = physical_crop(volume, px, crop_mm)

    # 侧性归一化
    volume = normalise_laterality(volume, plane, laterality)

    # 鲁棒归一化 (1st-99th percentile)
    v_low, v_high = np.percentile(volume, [1.0, 99.0])
    volume = np.clip(volume, v_low, v_high)
    denom = max(v_high - v_low, 1e-6)
    volume = (volume - v_low) / denom

    # 缩放到 target size
    resized = []
    for img in volume:
        r = cv2.resize(img, (image_size, image_size), interpolation=cv2.INTER_LINEAR)
        resized.append(r)
    return np.stack(resized, axis=0).astype(np.float32), px


# ---- 缓存切片采样 ----
def sample_cache_slices(volume, n_cache=9, center_pct=(0.2, 0.8)):
    """从 volume 的 central 60% 区域均匀采样 n_cache 个切片。"""
    n_total = volume.shape[0]
    if n_total <= n_cache:
        indices = list(range(n_total))
        while len(indices) < n_cache:
            indices.append(indices[-1])
        return volume[np.array(indices)]

    low = int(center_pct[0] * (n_total - 1))
    high = int(center_pct[1] * (n_total - 1))
    if high <= low:
        low, high = 0, n_total - 1
    indices = np.unique(np.linspace(low, high, n_cache).astype(int))
    while len(indices) < n_cache:
        indices = np.append(indices, indices[-1])
    return volume[indices[:n_cache]]

print('DICOM I/O v4 ready.')


In [ ]:
# ============================================================
# v4: SlotHead + MultiViewModel — 支持诊断池化
# ============================================================

class SlotHead(nn.Module):
    """Per-diagnosis attention over MRI slots with anatomical priors."""

    def __init__(self, dim, n_slot, n_out, hidden=256, p=0.2):
        super().__init__()
        self.proj = nn.Sequential(
            nn.LayerNorm(dim), nn.Linear(dim, hidden), nn.GELU())
        self.slot_emb = nn.Parameter(torch.randn(n_slot, hidden) * 0.02)
        self.query = nn.Parameter(torch.randn(n_out, hidden) * 0.02)
        self.drop = nn.Dropout(p)
        self.out = nn.Linear(hidden, n_out)
        self.hidden = hidden

        prior = torch.zeros(n_out, n_slot)
        for target_name, slot_indices in SLOT_PRIORS.items():
            if target_name in TARGET_COLUMNS:
                prior[TARGET_COLUMNS.index(target_name), list(slot_indices)] = 0.55
        self.register_buffer("slot_prior", prior)

    def forward(self, x, mask):
        h = self.proj(x) + self.slot_emb                              # [B, S, H]
        attention = (
            torch.einsum("bsh,oh->bos", h, self.query)                # [B, n_out, S]
            / math.sqrt(self.hidden)
            + self.slot_prior.unsqueeze(0)
        )
        attention = attention.masked_fill(
            mask.unsqueeze(1) < 0.5, -1e4).softmax(-1)
        context = self.drop(torch.einsum("bos,bsh->boh", attention, h))
        return (context * self.out.weight.unsqueeze(0)).sum(-1) + self.out.bias


class MultiViewModel(nn.Module):
    """DINOv2 + SlotHead for multi-view knee MRI."""

    def __init__(self, dinov2_model, n_slots=6, cls_dim=384,
                 n_classes=12, slot_hidden=256, dropout=0.2,
                 unfreeze_layers=6):
        super().__init__()
        self.n_slots = n_slots
        self.cls_dim = cls_dim
        self.feature_dim = cls_dim * 3
        self.unfreeze_layers = unfreeze_layers

        self.dinov2 = dinov2_model
        n_blocks = len(self.dinov2.blocks)
        if unfreeze_layers > 0:
            for p in self.dinov2.parameters():
                p.requires_grad = False
            unfreeze_start = max(0, n_blocks - unfreeze_layers)
            for block in self.dinov2.blocks[unfreeze_start:]:
                for p in block.parameters():
                    p.requires_grad = True
            if hasattr(self.dinov2, 'norm'):
                for p in self.dinov2.norm.parameters():
                    p.requires_grad = True

        self.head = SlotHead(
            dim=self.feature_dim, n_slot=n_slots, n_out=n_classes,
            hidden=slot_hidden, p=dropout)

        self.register_buffer("mean", torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1))
        self.register_buffer("std", torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1))

    def _extract_features(self, x_3ch):
        if self.unfreeze_layers > 0:
            features = self.dinov2.forward_features(x_3ch)
        else:
            with torch.no_grad():
                features = self.dinov2.forward_features(x_3ch)
        cls = features[:, 0, :]
        patches = features[:, 1:, :]
        mean_p = patches.mean(dim=1)
        k = max(1, patches.shape[1] // 8)
        focal = patches.topk(k, dim=1).values.mean(dim=1)
        return torch.cat([cls, mean_p, focal], dim=1)

    def forward(self, images, mask):
        """images: [B, S, 3, H, W] uint8 or [B*W, S, 3, H, W] for TTA"""
        B, S = images.shape[:2]
        x = images.reshape(B * S, 3, images.shape[-2], images.shape[-1])
        x = x.float().div_(255.0)
        x = (x - self.mean) / self.std
        features = self._extract_features(x)
        features = features.reshape(B, S, -1)
        return self.head(features, mask)

    def train(self, mode=True):
        super().train(mode)
        self.dinov2.eval()
        return self


# ---- ★ Jitter TTA 增广视图 (0.91 notebook augment() 移植) ----
def tta_jitter(imgs, seed=AUG_SEED):
    """每窗口生成一个确定性增广视图。

    几何（旋转 ±AUG_ROT_DEG° / 缩放 +[0, AUG_SCALE] / 平移 ±AUG_SHIFT）+
    强度 ±AUG_INTENSITY，border 填充（0.91 同款）。
    固定种子 → 同一批输入每次生成相同增广，验证/测试/提交全程可复现。
    输入 [..., 3, H, W] uint8 → 输出同形状同 dtype。
    """
    lead = imgs.shape[:-3]
    x = imgs.reshape(-1, *imgs.shape[-3:]).float()
    n, dev = (x.shape[0], x.device)
    gen = torch.Generator(device=dev).manual_seed(int(seed) % (2 ** 63 - 1))

    rot = (torch.rand(n, device=dev, generator=gen) - 0.5) * 2 * (AUG_ROT_DEG * np.pi / 180)
    sc = 1.0 + torch.rand(n, device=dev, generator=gen) * AUG_SCALE
    tx = (torch.rand(n, device=dev, generator=gen) - 0.5) * 2 * AUG_SHIFT
    ty = (torch.rand(n, device=dev, generator=gen) - 0.5) * 2 * AUG_SHIFT
    cos, sin = (torch.cos(rot) / sc, torch.sin(rot) / sc)

    theta = torch.zeros(n, 2, 3, device=dev, dtype=torch.float32)
    theta[:, 0, 0], theta[:, 0, 1], theta[:, 0, 2] = (cos, -sin, tx)
    theta[:, 1, 0], theta[:, 1, 1], theta[:, 1, 2] = (sin, cos, ty)

    grid = F.affine_grid(theta, x.shape, align_corners=False)
    x = F.grid_sample(x, grid, mode='bilinear', padding_mode='border', align_corners=False)

    scale = 1.0 + (torch.rand(n, 1, 1, 1, device=dev, generator=gen) - 0.5) * 2 * AUG_INTENSITY
    x = (x * scale).clamp(0, 255)
    return x.reshape(*lead, *x.shape[-3:]).to(imgs.dtype)


def stack_views(logits_flat, B, W, n_orig):
    """TTA 视图分组: [V*B*W, C]（B-major：每研究 W 行连续，原始块在前）→ [B, V*W, C]。

    V=2（jitter 开启）时输出每研究 [前 W 行原始视图, 后 W 行增广视图]；
    n_orig=None（无 jitter）时即 [B, W, C]。
    ★ 不可用 reshape(B, -1, C) 直接切——行序是研究大循环，会跨研究串位。
    """
    if n_orig is None:
        return logits_flat.reshape(B, W, -1)
    return logits_flat.view(2, B, W, -1).permute(1, 0, 2, 3).reshape(B, 2 * W, -1)


# ---- ★ 诊断特异性 TTA 池化 ----
DIAG_POOL_IDX = {}
for target_name, mode in DIAG_POOL.items():
    if target_name in TARGET_COLUMNS:
        DIAG_POOL_IDX[TARGET_COLUMNS.index(target_name)] = mode


def diagnostic_pool(logits_views, pool_idx=None, n_orig=None):
    """对 [B, V, C] logits 应用诊断特异性池化。

    - max:           局部病灶保留最强信号窗口
    - top2:          ACL/MCL 取前2强窗口平均
    - mean:          弥漫性病变取全窗口平均（默认）
    - original_mean: 仅无 jitter 原始视图平均（Synovitis, 0.91 同款）

    jitter TTA 模式（n_orig 给定）: 前 n_orig 个视图为原始视图、其余为增广视图；
    先按窗口做视图平均（0.91 的 win_probs），再做 per-target 窗口池化。
    n_orig=None 时全部视图视为原始视图（original_mean ≡ mean，与旧版行为一致）。
    """
    if pool_idx is None:
        pool_idx = DIAG_POOL_IDX

    B, V, C = logits_views.shape
    if n_orig is not None:
        orig_probs = torch.sigmoid(logits_views[:, :n_orig])             # [B, W, C]
        probs = (orig_probs + torch.sigmoid(logits_views[:, n_orig:])) / 2  # 视图平均
    else:
        probs = torch.sigmoid(logits_views)
        orig_probs = probs

    result = probs.mean(dim=1)                          # [B, C] — 默认 mean

    for j, mode in pool_idx.items():
        x = probs[:, :, j]                             # [B, W]
        if mode == 'max':
            result[:, j] = x.max(dim=1).values
        elif mode == 'top2':
            result[:, j] = x.topk(min(2, x.shape[1]), dim=1).values.mean(dim=1)
        elif mode == 'original_mean':
            result[:, j] = orig_probs[:, :, j].mean(dim=1)

    return result  # [B, C]


if IS_MAIN:
    n_total = sum(p.numel() for p in SlotHead(1152, 6, 12).parameters())
    print(f'SlotHead params: {n_total/1e6:.3f}M')
    print(f'Diag pool targets: {list(DIAG_POOL_IDX.keys())}')
    print(f'Jitter TTA: {"ON" if CFG.get("tta_jitter", False) else "OFF"} '
          f'(rot ±{AUG_ROT_DEG:.0f}°, scale +{AUG_SCALE:.0%}, '
          f'shift ±{AUG_SHIFT:.0%}, intensity ±{AUG_INTENSITY:.0%})')
    print('Model v4 ready.')


In [ ]:
# ============================================================
# v5: Loss Functions — WeightedSoftBCELoss (置信度加权软 BCE) + FocalBCELoss (v4 遗留)
# ============================================================

class FocalBCELoss(nn.Module):
    """Focal Loss for binary classification — v4 遗留, v5 不再使用。

    v4 用它处理「硬伪标签 + 类别不平衡」; v5 改用融合软标签 + 置信度权重后,
    软 BCE 直接携带置信度, focal 的难例加权与权重机制重叠, 反而放大噪声。
    FL = -alpha * (1 - pt)^gamma * log(pt)
    """
    def __init__(self, alpha=0.25, gamma=2.0, reduction='mean'):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma
        self.reduction = reduction

    def forward(self, logits, targets, weight=None, mask=None):
        probs = torch.sigmoid(logits)
        bce = F.binary_cross_entropy_with_logits(logits, targets, reduction='none')
        pt = torch.where(targets > 0.5, probs, 1.0 - probs)
        focal_weight = (1.0 - pt) ** self.gamma
        alpha_weight = torch.where(targets > 0.5, self.alpha, 1.0 - self.alpha)
        loss = alpha_weight * focal_weight * bce
        if weight is not None:
            loss = loss * weight
        if mask is not None:
            loss = loss * mask
        if self.reduction == 'mean':
            denom = mask.sum() if mask is not None else loss.numel()
            return loss.sum() / max(denom, 1.0)
        elif self.reduction == 'sum':
            return loss.sum()
        return loss


class WeightedSoftBCELoss(nn.Module):
    """★ v5 主损失：置信度加权的软标签 BCE (teacher-student 训练目标)。

    软标签 = per-finding 融合概率 (文本提取器 × 公开集成 OOF, 见 v5_labels.csv):
      - 携带两个 teacher 的置信度与 inter-class 结构 (0.73 与 0.80 的区分度
        优于两个硬标签 1/1)
      - 权重 = 标签置信度: 文本提及 (高 conf) 与 text/oof 一致时 → 1.0,
        静默 → ~0.35 (弱拉取, 从不断言阴性)
      - gold 行 (若有) weight=1.0, mask 控制参与
    loss = mean(bce(logits, prob) * weight * mask) / sum(mask)
    """
    def forward(self, logits, prob_targets, weights, mask):
        bce = F.binary_cross_entropy_with_logits(
            logits, prob_targets, reduction='none')
        loss = bce * weights * mask
        denom = mask.sum() + 1e-8
        return loss.sum() / denom


if IS_MAIN:
    print('Loss functions v5 ready (WeightedSoftBCE primary; FocalBCE legacy).')

"""Auxiliary within-batch ranking; no extra image passes or random sampling."""
def trusted_pair_rank_loss(logits, states, soft_masks, margin=0.1):
    logits = logits.float()
    losses = []
    counts = []
    for j in range(logits.shape[1]):
        active = soft_masks[:, j] > 0
        pos = logits[(states[:, j] == 1) & active, j]
        neg = logits[(states[:, j] == -1) & active, j]
        counts.append(pos.numel() * neg.numel())
        if pos.numel() and neg.numel():
            losses.append(F.softplus(neg[None, :] - pos[:, None] + margin).mean())
    loss = torch.stack(losses).mean() if losses else logits.sum() * 0.0
    return loss, counts


In [ ]:
# ============================================================
# v5: MultiViewDataset — 6-slot clinical MRI (288px)
# ============================================================

class MultiViewDataset(Dataset):
    """Multi-view knee MRI dataset with 6 clinical slots.

    训练：随机 3-slice 窗口 + 数据增强标志
    验证/测试：固定中间窗口
    """

    def __init__(self, study_uids, slot_map, cache, mask_array,
                 labels_df, study_index, is_train=True):
        self.study_uids = list(study_uids)
        self.slot_map = slot_map
        self.cache = cache
        self.mask_array = mask_array
        self.labels_df = labels_df
        self.study_index = study_index
        self.is_train = is_train

        # Filter to cached studies
        valid_uids = [u for u in self.study_uids if u in self.study_index]
        if IS_MAIN and len(valid_uids) < len(self.study_uids):
            print(f'[Dataset] {len(self.study_uids) - len(valid_uids)} studies skipped (not in cache)')
        self.study_uids = valid_uids

    def __len__(self):
        return len(self.study_uids)

    def __getitem__(self, idx):
        uid = self.study_uids[idx]
        row_idx = self.study_index[uid]

        slots = torch.from_numpy(self.cache[row_idx].copy())  # [6, 9, H, W]
        mask = torch.from_numpy(self.mask_array[row_idx].copy())  # [6]

        n_slices = slots.shape[1]  # 9
        if self.is_train:
            max_start = n_slices - CFG['group_size']
            start = torch.randint(0, max_start + 1, (1,)).item() if max_start > 0 else 0
            window = slots[:, start:start + CFG['group_size']]  # [6, 3, H, W]
        else:
            # ★ 返回全部9切片用于7窗口TTA
            window = slots  # [6, 9, H, W]

        label_row = self.labels_df.loc[uid]

        if self.is_train:
            probs = torch.tensor(
                [float(label_row.get(c, 0.5)) for c in PROB_COLS], dtype=torch.float32)
            weights = torch.tensor(
                [float(label_row.get(c, 0.1)) for c in WEIGHT_COLS], dtype=torch.float32)
            soft_masks = torch.tensor(
                [float(label_row.get(c, 0.0)) for c in MASK_COLS], dtype=torch.float32)
            return {
                'slots': window, 'mask': mask,
                'prob_targets': probs, 'weights': weights, 'soft_masks': soft_masks,
                'rank_states': torch.tensor([label_row['rank_' + c] for c in TARGET_COLUMNS], dtype=torch.int8),
                'study_uid': uid,
            }
        else:
            labels = torch.tensor(
                [float(label_row.get(c, 0.0)) for c in TARGET_COLUMNS], dtype=torch.float32)
            val_masks = torch.tensor(
                [float(label_row.get(c, 0.0)) for c in MASK_COLS], dtype=torch.float32)
            return {
                'slots': window, 'mask': mask,
                'labels': labels, 'val_masks': val_masks,
                'study_uid': uid,
            }

print('Dataset v4 ready.')


In [ ]:
# ============================================================
# v5: Load metadata + v5 融合软标签 (teacher-student) + Gold 全部→验证
# ============================================================

comp_input = Path(CFG['comp_input'])
label_input = Path(CFG['label_input'])

# ---- Load competition metadata ----
train_meta = pd.read_csv(comp_input / 'train.csv')
train_meta['StudyInstanceUID'] = train_meta['StudyInstanceUID'].astype(str)
series_meta = pd.read_csv(comp_input / 'train_series.csv')
series_meta['StudyInstanceUID'] = series_meta['StudyInstanceUID'].astype(str)
series_meta['SeriesInstanceUID'] = series_meta['SeriesInstanceUID'].astype(str)

# ---- Split gold vs unlabeled ----
label_cols_present = [c for c in TARGET_COLUMNS if c in train_meta.columns]
has_all_labels = train_meta[label_cols_present].notna().all(axis=1)
gold_df = train_meta[has_all_labels].copy()
unlabeled_df = train_meta[~has_all_labels].copy()

# ★ v5: 全部 gold → 验证 (与 v4 相同, 保证与 v4 0.833 可比)
gold_studies = sorted(gold_df['StudyInstanceUID'].unique())
unlabeled_studies = sorted(unlabeled_df['StudyInstanceUID'].unique())

val_gold_uids = set(gold_studies)       # ★ 全部 gold → val
train_gold_uids = set()                 # ★ 训练不使用 gold

if IS_MAIN:
    print(f'Gold studies (all 12 labeled): {len(gold_studies)}')
    print(f'Unlabeled studies: {len(unlabeled_studies)}')
    print(f'★ v5 split: Train={len(train_gold_uids)} gold + all fused, Val={len(val_gold_uids)} gold')

# ---- Gold labels (val only) ----
gold_labels = gold_df[['StudyInstanceUID'] + label_cols_present].copy()
gold_labels = gold_labels.set_index('StudyInstanceUID')
for c in TARGET_COLUMNS:
    if c not in gold_labels.columns:
        gold_labels[c] = np.nan
gold_labels = gold_labels.apply(pd.to_numeric, errors='coerce')
n_pos_per_class = (gold_labels > 0).sum(axis=0)
if IS_MAIN:
    print(f'Gold positives per class: min={int(n_pos_per_class.min())}, '
          f'max={int(n_pos_per_class.max())}, mean={n_pos_per_class.mean():.1f}')

# ---- ★ v5 融合软标签 (本地 scripts/build_v5_labels.py 生成) ----
# prob_*:  融合概率 = per-finding 逻辑回归 (gold 上拟合) 混合
#          提取器 score (text teacher) × 公开 20 成员集成 OOF (image teacher)
# weight_*: 置信度权重 = (0.35+0.65·conf) × text/oof 一致性, gold 行 = 1.0
# mask_*:  1.0 (软标签全参与, 权重即置信度)
v5_label_file = label_input / 'v5_labels.csv'
if not v5_label_file.exists():
    raise FileNotFoundError(
        f'v5 labels not found: {v5_label_file} — '
        f'upload data/processed/v5_labels.csv as a Kaggle Dataset '
        f'(root dir) and mount it; or set CFG["label_input"]')

import hashlib, json
assert hashlib.sha256(v5_label_file.read_bytes()).hexdigest() == 'c13adffaabf4f8e518abb038282bb1aa09baac7652a9165e030710c457d0be6a', 'Wrong historical v5 labels'
fused_df = pd.read_csv(v5_label_file)
fused_df['StudyInstanceUID'] = fused_df['StudyInstanceUID'].astype(str)

fused_labels = fused_df[['StudyInstanceUID']].copy()
for c in PROB_COLS:
    fused_labels[c] = fused_df[c]
for c in WEIGHT_COLS:
    fused_labels[c] = fused_df[c]
for c in MASK_COLS:
    fused_labels[c] = fused_df[c]
fused_labels = fused_labels.set_index('StudyInstanceUID')
for c in PROB_COLS + WEIGHT_COLS + MASK_COLS:
    fused_labels[c] = pd.to_numeric(fused_labels[c], errors='coerce').fillna(
        0.5 if 'prob' in c else 0.1).astype(np.float32)

if IS_MAIN:
    n_missing = len(set(unlabeled_studies) - set(fused_labels.index))
    print(f'v5 labels: {len(fused_labels):,} rows; missing for unlabeled: {n_missing}')

# ---- Train labels (fused soft labels for all unlabeled studies) ----
all_train_rows = []
for uid in unlabeled_studies:
    if uid not in fused_labels.index:
        continue
    row = {'StudyInstanceUID': uid}
    for c in TARGET_COLUMNS:
        row[c] = 0.0
    for c in PROB_COLS:
        row[c] = float(fused_labels.loc[uid, c])
    for c in WEIGHT_COLS:
        row[c] = float(fused_labels.loc[uid, c])
    for c in MASK_COLS:
        row[c] = float(fused_labels.loc[uid, c])
    row['is_gold'] = False
    all_train_rows.append(row)

train_labels = pd.DataFrame(all_train_rows).set_index('StudyInstanceUID')

# ---- Val labels (gold only) ----
val_rows = []
for uid in val_gold_uids:
    if uid not in gold_labels.index:
        continue
    row = {'StudyInstanceUID': uid}
    for c in TARGET_COLUMNS:
        raw = gold_labels.loc[uid, c]
        is_labeled = not pd.isna(raw)
        row[c] = float(raw) if is_labeled else 0.0
        row[f'mask_{c}'] = 1.0 if is_labeled else 0.0
    row['is_gold'] = True
    val_rows.append(row)
val_labels = pd.DataFrame(val_rows).set_index('StudyInstanceUID')

if IS_MAIN:
    n_train = len(train_labels)
    print(f'\nTrain: {n_train:,} studies (v5 fused soft labels)')
    print(f'Val:   {len(val_labels):,} studies (all gold-labeled)')
    val_labeled = val_labels[MASK_COLS].sum(axis=0) if len(val_labels) > 0 else pd.Series(0, index=MASK_COLS)
    print(f'  Labeled per class: min={int(val_labeled.min())}, mean={val_labeled.mean():.1f}')

    # 标签分布 (正类率 / 平均权重) — 与本地融合报告对照
    dist = pd.DataFrame({
        'pos_rate': [(train_labels[f'prob_{c}'] > 0.5).mean() for c in TARGET_COLUMNS],
        'mean_prob': [train_labels[f'prob_{c}'].mean() for c in TARGET_COLUMNS],
        'mean_weight': [train_labels[f'weight_{c}'].mean() for c in TARGET_COLUMNS],
    }, index=TARGET_COLUMNS).round(3)
    print(dist.to_string())

_trust_path = Path(CFG['trust_input']) / 'stage3d_trust.npz'
assert hashlib.sha256(_trust_path.read_bytes()).hexdigest() == '92600b0df5bee567502240533fd8e47281225ca3a55f429b5e57cd74fc6515f6', 'Wrong ranking asset'
_trust = np.load(_trust_path, allow_pickle=False)
assert list(_trust['targets']) == TARGET_COLUMNS
assert _trust['state'].shape == (len(_trust['ids']), len(TARGET_COLUMNS))
assert np.isin(_trust['state'], [-1, 0, 1]).all()
_rank_df = pd.DataFrame(_trust['state'], index=_trust['ids'].astype(str), columns=TARGET_COLUMNS)
assert _rank_df.index.is_unique
assert set(_rank_df.index) == set(unlabeled_studies) == set(train_labels.index)
assert not set(_rank_df.index) & set(val_labels.index)
_active_classes = []
for _c in TARGET_COLUMNS:
    _counts = _rank_df[_c].value_counts()
    if min(_counts.get(1, 0), _counts.get(-1, 0)) < CFG['rank_min_cases']:
        _rank_df[_c] = 0
    else:
        _active_classes.append(_c)
    train_labels['rank_' + _c] = _rank_df[_c].reindex(train_labels.index)
assert np.isfinite(train_labels[PROB_COLS + WEIGHT_COLS + MASK_COLS].values).all()
_preflight = {'config': CFG, 'label_sha256': 'c13adffaabf4f8e518abb038282bb1aa09baac7652a9165e030710c457d0be6a', 'trust_sha256': '92600b0df5bee567502240533fd8e47281225ca3a55f429b5e57cd74fc6515f6',
              'n_train': len(train_labels), 'n_gold': len(val_labels), 'rank_classes': _active_classes,
              'teacher_provenance': 'Crossfit training manifest unavailable; not independently verified'}
Path(CFG['output_dir']).mkdir(parents=True, exist_ok=True)
(Path(CFG['output_dir']) / 'phase3d_preflight.json').write_text(json.dumps(_preflight, indent=2))
print('Ranking classes:', _active_classes)


In [ ]:
# ============================================================
# v4: Build RAM Cache — 并行 DICOM 读取 + 空间排序 + 物理裁剪 + 侧性归一化
# ============================================================

dicom_root = Path(CFG['comp_input']) / CFG['dicom_subdir']
print(f'DICOM root: {dicom_root}')

# ---- Build slot mapping ----
slot_map, study_series_map = build_study_slot_map(series_meta, dicom_root)

# ---- ★ 侧性检测 ----
# 快速扫描：每个 study 只读一个 DICOM header 来获取 Laterality
all_needed_uids = set(train_labels.index) | set(val_labels.index)
print(f'Studies to cache: {len(all_needed_uids)}')

needed_slot_map = {uid: slot_map[uid] for uid in all_needed_uids if uid in slot_map}

# ★ 快速侧性检测：每个 study 扫描一个 DICOM header
def _detect_laterality_fast(needed_slot_map):
    """为每个 study 快速检测侧性（只读每个 study 第一个有效 series 的 header）。"""
    laterality_map = {}
    for study_uid, study_slots in needed_slot_map.items():
        lat = None
        for slot_name, slot_info in study_slots.items():
            if slot_info is None:
                continue
            series_dir = Path(slot_info['dir']) if 'dir' in slot_info else None
            if series_dir is None or not series_dir.exists():
                continue
            dcm_files = _list_dcm_files(series_dir)
            if not dcm_files:
                continue
            try:
                ds = pydicom.dcmread(
                    str(series_dir / dcm_files[0]), stop_before_pixels=True, force=True,
                    specific_tags=['Laterality', 'ImageLaterality', 'ImagePositionPatient'])
                # 优先 DICOM Laterality 标签
                for tag_name in ['Laterality', 'ImageLaterality']:
                    val = getattr(ds, tag_name, None)
                    if val is not None:
                        val = str(val).strip().upper()
                        if val and val[0] in ('L', 'R'):
                            lat = val[0]
                            break
                if lat is not None:
                    break
                # Fallback: ImagePositionPatient 几何推断 (LPS: +x = 患者左侧)
                ipp = getattr(ds, 'ImagePositionPatient', None)
                if ipp is not None and len(ipp) >= 1:
                    try:
                        x = float(str(ipp[0]).split('\\')[0].split('|')[0])
                        if abs(x) >= 5.0:
                            lat = 'R' if x < 0 else 'L'
                            break
                    except Exception:
                        pass
            except Exception:
                continue
        laterality_map[study_uid] = lat
    return laterality_map

t_lat = time.time()
laterality_map = _detect_laterality_fast(needed_slot_map)
n_lat = sum(1 for v in laterality_map.values() if v is not None)
n_right = sum(1 for v in laterality_map.values() if v == 'R')
if IS_MAIN:
    print(f'Laterality detected: {n_lat}/{len(laterality_map)} studies '
          f'({n_lat/max(len(laterality_map),1)*100:.1f}%), '
          f'R={n_right}, L={n_lat-n_right}, '
          f'({time.time()-t_lat:.1f}s)')

# ---- Pre-allocate cache ----
n_cache_studies = len(needed_slot_map)
cache_shape = (n_cache_studies, N_SLOT, CFG['cache_slices'], CFG['image_size'], CFG['image_size'])

SLOT_CACHE = np.zeros(cache_shape, dtype=np.uint8)
SLOT_MASK = np.zeros((n_cache_studies, N_SLOT), dtype=np.float32)
study_index = {}

print(f'Cache: {cache_shape} = {SLOT_CACHE.nbytes / 1024**3:.2f} GB uint8')

# ---- ★ 并行 DICOM 读取 ----
def _read_slot_job(args):
    """单个 slot 的读取任务（用于 ThreadPoolExecutor）"""
    row_idx, slot_idx, slot_name, plane, slot_info, laterality = args
    if slot_info is None:
        return row_idx, slot_idx, None

    series_dir = Path(slot_info['dir']) if 'dir' in slot_info else None
    if series_dir is None or not series_dir.exists():
        return row_idx, slot_idx, None

    try:
        volume, px = read_series_volume(
            str(series_dir), plane=plane, laterality=laterality,
            image_size=CFG['image_size'], crop_mm=CFG['crop_mm'])
        if volume is None or volume.shape[0] < 3:
            return row_idx, slot_idx, None

        sampled = sample_cache_slices(
            volume, n_cache=CFG['cache_slices'], center_pct=CFG['center_pct'])
        sampled_uint8 = (sampled * 255).clip(0, 255).round().astype(np.uint8)
        return row_idx, slot_idx, sampled_uint8
    except Exception:
        return row_idx, slot_idx, None


# ---- Fill cache ----
t_cache = time.time()
sorted_uids = sorted(needed_slot_map)

for row_idx, study_uid in enumerate(sorted_uids):
    study_index[study_uid] = row_idx

# 收集所有读取任务
jobs = []
for row_idx, study_uid in enumerate(sorted_uids):
    study_slots = needed_slot_map[study_uid]
    lat = laterality_map.get(study_uid)  # ★ 侧性归一化
    for slot_idx, (slot_name, plane, fluid, fatsat) in enumerate(SLOTS):
        slot_info = study_slots.get(slot_name)
        if slot_info is not None:
            jobs.append((row_idx, slot_idx, slot_name, plane, slot_info, lat))

print(f'Decoding {len(jobs)} slot-series (parallel, {CFG["pix_threads"]} threads)...')

completed = 0
failed = 0
with ThreadPoolExecutor(max_workers=CFG['pix_threads']) as pool:
    for row_idx, slot_idx, result in pool.map(_read_slot_job, jobs):
        completed += 1
        if result is not None:
            SLOT_CACHE[row_idx, slot_idx] = result
            SLOT_MASK[row_idx, slot_idx] = 1.0
        else:
            failed += 1

        if completed % 2000 == 0:
            elapsed = time.time() - t_cache
            pct = completed / len(jobs) * 100
            eta = (elapsed / completed) * (len(jobs) - completed) / 60
            print(f'  [{completed}/{len(jobs)}] {pct:.0f}% | {elapsed:.0f}s | ~{eta:.0f}min left', flush=True)

cache_time = time.time() - t_cache
total_series = int(SLOT_MASK.sum())
print(f'\nCache built: {n_cache_studies} studies, {total_series} series, '
      f'{SLOT_CACHE.nbytes / 1024**3:.1f} GB in {cache_time:.0f}s')
print(f'  Avg slots/study: {total_series/max(n_cache_studies,1):.1f}')
print(f'  Failed reads: {failed}')
print(f'  ★ Physical crop: {CFG["crop_mm"]}mm | Laterality: {n_right}R/{n_lat-n_right}L | Threads: {CFG["pix_threads"]}')

# ★ v5: 自蒸馏已由 v5 融合软标签取代 (09_load_data 中的 fused labels)，
# 无需 v3 checkpoint 重新打标。

gc.collect()


In [ ]:
# ============================================================
# v4: DataLoaders
# ============================================================

train_ds = MultiViewDataset(
    study_uids=list(train_labels.index),
    slot_map=needed_slot_map,
    cache=SLOT_CACHE, mask_array=SLOT_MASK,
    labels_df=train_labels, study_index=study_index,
    is_train=True,
)

val_ds = MultiViewDataset(
    study_uids=list(val_labels.index),
    slot_map=needed_slot_map,
    cache=SLOT_CACHE, mask_array=SLOT_MASK,
    labels_df=val_labels, study_index=study_index,
    is_train=False,
)

# ---- 随机种子: DataLoader shuffle 顺序与 worker 的确定性 ----
#   每 seed 一个会话 → 不同训练顺序 (自集成成员独立性的主要来源之一)
_gen = torch.Generator()
_gen.manual_seed(CFG['seed'])

def seed_worker(worker_id):
    worker_seed = CFG['seed'] + worker_id
    random.seed(worker_seed)
    np.random.seed(worker_seed)
    torch.manual_seed(worker_seed)

train_loader = DataLoader(
    train_ds, batch_size=CFG['batch_size'], shuffle=True,
    num_workers=CFG['num_workers'], pin_memory=True, drop_last=True,
    generator=_gen, worker_init_fn=seed_worker,
)

val_loader = DataLoader(
    val_ds, batch_size=CFG['batch_size'], shuffle=False,
    num_workers=CFG['num_workers'], pin_memory=True,
)

if IS_MAIN:
    print(f'Train: {len(train_ds)} studies → {len(train_loader)} batches × {CFG["batch_size"]}')
    print(f'Val:   {len(val_ds)} studies → {len(val_loader)} batches × {CFG["batch_size"]}')


In [ ]:
# ============================================================
# v4: Training & Validation — Focal Loss + 诊断池化 + EMA
# ============================================================

# ---- EMA ----
class EMAModel:
    """Exponential Moving Average of model weights."""
    def __init__(self, model, decay=0.999):
        self.model = model
        self.decay = decay
        self.shadow = {}
        self.backup = {}
        self._register()

    def _register(self):
        for name, param in self.model.named_parameters():
            if param.requires_grad:
                self.shadow[name] = param.data.clone()

    def update(self):
        for name, param in self.model.named_parameters():
            if param.requires_grad:
                self.shadow[name].mul_(self.decay).add_(param.data, alpha=1.0 - self.decay)

    def apply_shadow(self):
        """Replace model params with EMA params (for validation)."""
        for name, param in self.model.named_parameters():
            if param.requires_grad:
                self.backup[name] = param.data.clone()
                param.data.copy_(self.shadow[name])

    def restore(self):
        """Restore original model params."""
        for name, param in self.model.named_parameters():
            if param.requires_grad:
                param.data.copy_(self.backup[name])
        self.backup.clear()

    def state_dict(self):
        return {'decay': self.decay, 'shadow': self.shadow}

    def load_state_dict(self, state_dict):
        self.decay = state_dict['decay']
        self.shadow = state_dict['shadow']


# ---- Training ----
def train_epoch(model, loader, optimizer, criterion, scaler, epoch, ema=None):
    model.train()
    total_loss = 0.0
    n_batches = 0
    optimizer.zero_grad()
    use_amp = scaler is not None
    grad_accum = CFG.get('grad_accum_steps', 1)
    rank_total = bce_total = active_batches = 0
    pair_counts = np.zeros(len(TARGET_COLUMNS), dtype=np.int64)

    for bi, batch in enumerate(loader):
        slots = batch['slots'].to(DEVICE, non_blocking=True)
        mask = batch['mask'].to(DEVICE, non_blocking=True)
        prob_targets = batch['prob_targets'].to(DEVICE, non_blocking=True)
        weights = batch['weights'].to(DEVICE, non_blocking=True)
        soft_masks = batch['soft_masks'].to(DEVICE, non_blocking=True)

        with torch.amp.autocast('cuda', enabled=use_amp):
            logits = model(slots, mask)
            bce_loss = criterion(logits, prob_targets, weights, soft_masks)
            rank_loss, counts = trusted_pair_rank_loss(
                logits, batch['rank_states'].to(DEVICE), soft_masks, CFG['rank_margin'])
            loss = bce_loss + CFG['rank_lambda'] * rank_loss
            bce_total += bce_loss.detach().item()
            rank_total += rank_loss.detach().item()
            pair_counts += np.asarray(counts)
            active_batches += int(sum(counts) > 0)
            loss = loss / grad_accum

        if use_amp:
            scaler.scale(loss).backward()
        else:
            loss.backward()

        if (bi + 1) % grad_accum == 0:
            if use_amp:
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), CFG['grad_clip'])
                scaler.step(optimizer)
                scaler.update()
            else:
                torch.nn.utils.clip_grad_norm_(model.parameters(), CFG['grad_clip'])
                optimizer.step()
            optimizer.zero_grad()

            if ema is not None:
                ema.update()

        total_loss += loss.item() * grad_accum
        n_batches += 1

        if IS_MAIN and bi % 50 == 0:
            slots_present = mask.sum(dim=1).mean().item()
            print(f'  Epoch {epoch:3d} [{bi:4d}/{len(loader):4d}] '
                  f'loss={loss.item()*grad_accum:.4f} | slots={slots_present:.1f}/6',
                  flush=True)

    record = {'epoch': epoch, 'bce': bce_total / max(n_batches, 1),
              'rank_loss': rank_total / max(n_batches, 1),
              'weighted_rank_loss': CFG['rank_lambda'] * rank_total / max(n_batches, 1),
              'active_batches': active_batches, 'batches': n_batches,
              'pairs_by_class': dict(zip(TARGET_COLUMNS, pair_counts.tolist()))}
    with (Path(CFG['output_dir']) / 'ranking_history.jsonl').open('a') as handle:
        handle.write(json.dumps(record) + '\n')
    print('Ranking diagnostics:', record)
    return total_loss / max(n_batches, 1)


# ---- Validation (with 7-window TTA + diagnostic pooling) ----
@torch.no_grad()
def validate_epoch(model, loader, criterion_hard):
    model.eval()

    n_windows = CFG['cache_slices'] - CFG['group_size'] + 1  # 7

    all_probs, all_labels, all_masks, all_uids = [], [], [], []
    total_loss, n_batches = 0.0, 0

    for batch in loader:
        slots_full = batch['slots'].to(DEVICE, non_blocking=True)  # [B, 6, 9, H, W]
        mask = batch['mask'].to(DEVICE, non_blocking=True)          # [B, 6]
        labels = batch['labels'].to(DEVICE, non_blocking=True)      # [B, 12]
        val_masks = batch['val_masks'].to(DEVICE, non_blocking=True)  # [B, 12]
        uids = batch['study_uid']

        B = slots_full.shape[0]

        # ★ 损失只在中间窗口上算（省算力）
        mid_start = (CFG['cache_slices'] - CFG['group_size']) // 2  # 3
        slots_mid = slots_full[:, :, mid_start:mid_start + CFG['group_size']]  # [B, 6, 3, H, W]
        logits_mid = model(slots_mid, mask)
        active = val_masks > 0.5
        if active.any():
            loss_val = F.binary_cross_entropy_with_logits(
                logits_mid[active], labels[active], reduction='mean')
            total_loss += loss_val.item()
        n_batches += 1

        # ★ 7窗口 TTA + 诊断池化：单次批量前向传播
        #   B-major 布局（每研究 7 窗口连续），stack_views 分组——防止跨研究串位
        #   jitter TTA (0.91 移植): 每窗口额外 1 个确定性增广视图 → 视图平均 → per-target 窗口池化
        windows = torch.stack(
            [slots_full[:, :, w:w + CFG['group_size']] for w in range(n_windows)], dim=1)  # [B, 7, 6, 3, H, W]
        slots_flat = windows.reshape(B * n_windows, *windows.shape[2:])  # [B*7, ...] B-major
        mask_flat = mask.unsqueeze(1).expand(B, n_windows, -1).reshape(B * n_windows, -1)
        if CFG.get('tta_jitter', False):
            slots_flat = torch.cat([slots_flat, tta_jitter(slots_flat)], dim=0)  # [2*B*7, ...] 原始块在前
            mask_flat = mask_flat.repeat(2, 1)
            n_orig = n_windows
        else:
            n_orig = None
        logits_flat = model(slots_flat, mask_flat)      # [V*B*7, C]
        logits_views = stack_views(logits_flat, B, n_windows, n_orig)  # [B, V*7, C]
        probs_tta = diagnostic_pool(logits_views, n_orig=n_orig)  # [B, C]

        all_probs.append(probs_tta.cpu())
        all_labels.append(labels.cpu())
        all_masks.append(val_masks.cpu())
        all_uids.extend(uids)

    probs_all = torch.cat(all_probs, dim=0).numpy()
    labels_all = torch.cat(all_labels, dim=0).numpy()
    masks_all = torch.cat(all_masks, dim=0).numpy()

    # Per-class AUC on labeled studies only
    aucs = []
    per_class = {}
    for i, c in enumerate(TARGET_COLUMNS):
        labeled_idx = masks_all[:, i] > 0.5
        n_labeled = int(labeled_idx.sum())
        metrics = {'auc': float('nan'), 'n_pos': 0, 'n_total': n_labeled}

        if n_labeled > 1:
            y_true = labels_all[:, i][labeled_idx]
            y_prob = probs_all[:, i][labeled_idx]
            n_pos = int(y_true.sum())
            metrics['n_pos'] = n_pos
            if n_pos > 0 and n_pos < n_labeled:
                try:
                    metrics['auc'] = float(roc_auc_score(y_true, y_prob))
                    aucs.append(metrics['auc'])
                except Exception:
                    pass
        per_class[c] = metrics

    return {
        'loss': total_loss / max(n_batches, 1),
        'macro_auc': float(np.mean(aucs)) if aucs else 0.0,
        'per_class': per_class,
        'probs': probs_all, 'labels': labels_all,
        'uids': all_uids,
    }


def print_validation_summary(val_metrics):
    print(f'\n  {"Class":<20s} {"AUC":>7s} {"Pos":>5s}')
    print(f'  {"-"*20} {"-"*7} {"-"*5}')
    for c in TARGET_COLUMNS:
        m = val_metrics['per_class'][c]
        auc_str = f'{m["auc"]:.3f}' if not math.isnan(m['auc']) else '  N/A  '
        print(f'  {c:<20s} {auc_str:>7s} {m["n_pos"]:5d}')
    print(f'  {"-"*20} {"-"*7} {"-"*5}')
    print(f'  {"Macro AUC":<20s} {val_metrics["macro_auc"]:7.3f}')
    print()


def save_validation_report(val_metrics, output_dir, epoch=None, is_best=False):
    out = Path(output_dir)
    rows = []
    for c in TARGET_COLUMNS:
        m = val_metrics['per_class'][c]
        rows.append({'class': c, 'auc': m['auc'], 'n_pos': m['n_pos'],
                     'n_total': m['n_total']})
    report_df = pd.DataFrame(rows)
    report_df['macro_auc'] = val_metrics['macro_auc']
    report_df['val_loss'] = val_metrics['loss']

    tag = '_best' if is_best else f'_epoch{epoch}'
    report_df.to_csv(out / f'validation_report{tag}.csv', index=False)
    return report_df

print('Training functions v4 ready (EMA + FocalLoss).')


In [ ]:
# ============================================================
# v5: Build model, optimizer, scheduler + EMA — 288px + WeightedSoftBCE
# ============================================================

if IS_MAIN: print('Loading DINOv2 backbone...')

dinov2_backbone = timm.create_model(
    CFG['dinov2_variant'], pretrained=False, num_classes=0,
    img_size=CFG['image_size'],
)

# ★ 竞赛禁网，从本地 Kaggle Dataset 加载预训练权重
weights_path = Path(CFG.get('dinov2_weights', ''))
if weights_path.exists():
    state_dict = torch.load(weights_path, map_location='cpu', weights_only=True)

    # ★ DINOv2 原生 img_size=518 → pos_embed [1, 1370, 384] (37×37 grid)
    # v5 模型 img_size=288 → grid 288//14=20 → pos_embed [1, 401, 384]，需要插值
    if 'pos_embed' in state_dict:
        pos_ckpt = state_dict['pos_embed']          # [1, N_ckpt, dim]
        pos_model = dinov2_backbone.pos_embed.data   # [1, N_model, dim]
        if pos_ckpt.shape != pos_model.shape:
            cls_ckpt = pos_ckpt[:, :1, :]             # CLS token 保留
            patch_ckpt = pos_ckpt[:, 1:, :]           # patch tokens

            grid_ckpt = int(math.isqrt(patch_ckpt.shape[1]))
            grid_model = int(math.isqrt(pos_model.shape[1] - 1))

            patch_ckpt = patch_ckpt.reshape(1, grid_ckpt, grid_ckpt, -1).permute(0, 3, 1, 2)
            # bicubic 插值 → 目标 grid
            patch_interp = F.interpolate(
                patch_ckpt, size=(grid_model, grid_model), mode='bicubic',
                antialias=True)
            patch_interp = patch_interp.permute(0, 2, 3, 1).reshape(1, -1, pos_model.shape[-1])
            state_dict['pos_embed'] = torch.cat([cls_ckpt, patch_interp], dim=1)
            if IS_MAIN:
                print(f'  pos_embed interpolated: [{grid_ckpt}×{grid_ckpt}] → [{grid_model}×{grid_model}]')

    dinov2_backbone.load_state_dict(state_dict, strict=True)
    if IS_MAIN: print(f'  DINOv2 pretrained weights loaded: {weights_path}')
elif IS_MAIN:
    print(f'  WARNING: DINOv2 weights not found at {weights_path} — using random init!')

model = MultiViewModel(
    dinov2_model=dinov2_backbone,
    n_slots=N_SLOT,
    cls_dim=CFG['cls_dim'],
    n_classes=CFG['num_classes'],
    slot_hidden=CFG['slot_hidden'],
    dropout=CFG['dropout'],
    unfreeze_layers=CFG['unfreeze_layers'],
).to(DEVICE)

if N_GPUS > 1:
    model = nn.DataParallel(model)
    if IS_MAIN: print(f'[Model] DataParallel across {N_GPUS} GPUs')

# Separate LR
backbone_params, head_params = [], []
for name, p in model.named_parameters():
    if not p.requires_grad:
        continue
    if 'dinov2' in name:
        backbone_params.append(p)
    else:
        head_params.append(p)

optimizer = torch.optim.AdamW([
    {'params': backbone_params, 'lr': CFG['backbone_lr']},
    {'params': head_params, 'lr': CFG['lr']},
], weight_decay=CFG['weight_decay'])

# ★ v5: 置信度加权软 BCE — 融合软标签 (text×OOF teacher) 直接作为训练目标
criterion = WeightedSoftBCELoss()

scheduler = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(
    optimizer, T_0=CFG['lr_t0'], T_mult=CFG['lr_t_mult'],
    eta_min=CFG['lr_eta_min'])

scaler = torch.amp.GradScaler('cuda') if CFG['mixed_precision'] else None

# ★ EMA
ema = EMAModel(model.module if N_GPUS > 1 else model, decay=CFG['ema_decay'])

if IS_MAIN:
    n_backbone = sum(p.numel() for p in backbone_params)
    n_head = sum(p.numel() for p in head_params)
    print(f'Optimizer: backbone {n_backbone/1e6:.1f}M @ lr={CFG["backbone_lr"]}')
    print(f'           head     {n_head/1e6:.1f}M @ lr={CFG["lr"]}')
    print(f'Loss: WeightedSoftBCE (confidence-weighted fused soft labels)')
    print(f'EMA: decay={CFG["ema_decay"]}')
    print('Model v5 ready.')


In [ ]:
# ============================================================
# v4: Sanity Check — 管线完整性检查
# ============================================================

if IS_MAIN:
    print('=' * 50)
    print('SANITY CHECK')
    print('=' * 50)

    # 缓存形状
    print(f'Cache: {SLOT_CACHE.shape} | {SLOT_CACHE.dtype} | {SLOT_CACHE.nbytes/1024**3:.2f} GB')
    print(f'Mask:  {SLOT_MASK.shape} | slots/study: {SLOT_MASK.sum(axis=1).mean():.1f}')

    # 训练/验证集
    print(f'Train: {len(train_ds):,} studies | Val: {len(val_ds):,} studies')

    # 前向传播测试
    batch = next(iter(train_loader))
    slots = batch['slots'].to(DEVICE)
    mask = batch['mask'].to(DEVICE)
    with torch.no_grad():
        logits = model.module(slots, mask) if N_GPUS > 1 else model(slots, mask)
    print(f'Forward: {slots.shape} → {logits.shape} | '
          f'logits range [{logits.min().item():.3f}, {logits.max().item():.3f}]')
    print(f'  Mean sigmoid: {torch.sigmoid(logits).mean().item():.3f}')

    # GPU 内存
    if torch.cuda.is_available():
        mem = torch.cuda.memory_allocated() / 1024**3
        print(f'GPU memory: {mem:.2f} GB allocated')

    print('Sanity check PASSED.')


In [ ]:
(Path(CFG['output_dir']) / 'ranking_history.jsonl').write_text('')
# ============================================================
# v5: Training Loop — EMA + Early Stopping + 墙钟保护 + 最佳模型保存
# ============================================================

output_dir = Path(CFG['output_dir'])
(output_dir / 'checkpoints').mkdir(parents=True, exist_ok=True)

best_auc = 0.0
best_epoch = 0
patience_counter = 0
history = []

criterion_val = nn.BCEWithLogitsLoss(reduction='mean')

print(f'Training: {len(train_ds)} studies, {CFG["epochs"]} epochs')
print(f'  Effective batch = {CFG["batch_size"]} × {N_GPUS} GPU × {CFG["grad_accum_steps"]} accum = '
      f'{CFG["batch_size"] * max(N_GPUS, 1) * CFG["grad_accum_steps"]}')
print(f'  WeightedSoftBCE (confidence-weighted) | EMA({CFG["ema_decay"]})')
print(f'  ★ 288px / Physical crop: {CFG["crop_mm"]}mm | Laterality norm | Spatial ordering')
print(f'  ★ Wall-clock budget: {CFG["max_train_minutes"]}min (Kaggle 9h 会话上限)')
print(f'  ★ Seed: {CFG["seed"]} → checkpoint {CKPT_NAME} (换 seed 重跑 = 新集成成员)')
print()

t_start = time.time()

for epoch in range(1, CFG['epochs'] + 1):
    t_epoch = time.time()

    # Train
    train_loss = train_epoch(
        model, train_loader, optimizer, criterion, scaler, epoch, ema=ema)
    scheduler.step()

    # Validate (with EMA weights)
    if ema is not None:
        ema.apply_shadow()
    val_metrics = validate_epoch(model, val_loader, criterion_val)
    if ema is not None:
        ema.restore()

    val_loss = val_metrics['loss']
    val_auc = val_metrics['macro_auc']

    elapsed = time.time() - t_epoch
    eta_total = (time.time() - t_start) / epoch * (CFG['epochs'] - epoch) / 60

    history.append({
        'epoch': epoch, 'train_loss': train_loss,
        'val_loss': val_loss, 'macro_auc': val_auc,
    })

    if IS_MAIN:
        print(f'--- Epoch {epoch:3d} | '
              f'train_loss={train_loss:.4f} | val_loss={val_loss:.4f} | '
              f'val_auc={val_auc:.4f} | {elapsed:.0f}s | ~{eta_total:.0f}min left ---')
        print_validation_summary(val_metrics)

    # Save best
    is_best = val_auc > best_auc
    if is_best:
        best_auc = val_auc
        best_epoch = epoch
        patience_counter = 0

        # 获取实际模型（去掉 DataParallel wrapper）
        save_model = model.module if N_GPUS > 1 else model
        ckpt = {
            'epoch': epoch,
            'model': save_model.state_dict(),
            'ema': ema.state_dict() if ema else None,
            'auc': best_auc,
            'config': CFG,
            'targets': TARGET_COLUMNS,
            'slots': SLOTS,
        }
        torch.save(ckpt, output_dir / 'checkpoints' / CKPT_NAME)

        save_validation_report(val_metrics, output_dir, epoch=epoch, is_best=True)
        print(f'  ★ Best model saved (epoch={epoch}, AUC={best_auc:.4f})')
    else:
        patience_counter += 1

    # 定期保存
    if epoch % 10 == 0:
        save_model = model.module if N_GPUS > 1 else model
        torch.save({
            'epoch': epoch,
            'model': save_model.state_dict(),
            'ema': ema.state_dict() if ema else None,
            'auc': val_auc,
            'config': CFG, 'targets': TARGET_COLUMNS, 'slots': SLOTS,
        }, output_dir / 'checkpoints' / f'model_epoch{epoch}.pt')

    # Early stopping
    if patience_counter >= CFG['early_stop_patience']:
        print(f'Early stopping at epoch {epoch} (patience={CFG["early_stop_patience"]})')
        break

    # ★ 墙钟保护: 训练超过预算即优雅停止 (best checkpoint 已在上面保存)
    elapsed_min = (time.time() - t_start) / 60
    if elapsed_min > CFG['max_train_minutes']:
        print(f'Wall-clock budget reached ({elapsed_min:.0f}min > '
              f'{CFG["max_train_minutes"]}min) — stopping after epoch {epoch}')
        break

# ---- Save training history ----
hist_df = pd.DataFrame(history)
hist_df.to_csv(output_dir / 'training_history.csv', index=False)

total_time = time.time() - t_start
print(f'\n{"="*60}')
print(f'Training complete: {total_time/60:.0f}min | Best AUC={best_auc:.4f} @ epoch {best_epoch}')
print(f'Best model: {output_dir / "checkpoints" / CKPT_NAME}')
print(f'{"="*60}')


### 关于阈值：无需调整

本竞赛指标为 **macro ROC-AUC** —— 只看排序，对单调变换不变：

- submission.csv 直接写模型输出概率，**不需要**任何阈值
- 验证集上 per-class 的「最优阈值」只反映该类的校准偏差，供诊断用
- 若某类验证集最优阈值严重偏离 0.5（<0.3 或 >0.7），说明该类存在系统性偏差，可在下一版中通过标签先验或 specialist 修正，但提交分数不受阈值影响


In [ ]:
# ============================================================
# v5: Test Set Inference + Submission.csv + Gold Validation
# ============================================================
#
# 1. 在 gold 验证集上做完整 TTA + 诊断池化，计算真实 AUC
# 2. 在 test 集上推理，生成 submission.csv
# ============================================================

print('=' * 60)
print('GOLD VALIDATION + TEST INFERENCE')
print('=' * 60)

# ---- 加载最佳 checkpoint ----
best_ckpt_path = output_dir / 'checkpoints' / CKPT_NAME
if not best_ckpt_path.exists():
    raise FileNotFoundError(
        f'Best model checkpoint not found: {best_ckpt_path}\n'
        f'Training must produce {CKPT_NAME} (best validation AUC). '
        'Check that at least one epoch completed and saved a best model.')

print(f'Loading checkpoint: {best_ckpt_path}')
ckpt = torch.load(best_ckpt_path, map_location='cpu', weights_only=False)

# ★ seed 交叉验证: 防止下载归档时混用不同会话的产物
ckpt_seed = (ckpt.get('config') or {}).get('seed', '?')
print(f'  Checkpoint seed: s{ckpt_seed} | 本会话 seed: s{CFG["seed"]}')

# ---- Build inference model ----
infer_backbone = timm.create_model(
    CFG['dinov2_variant'], pretrained=False, num_classes=0, img_size=CFG['image_size'])

infer_model = MultiViewModel(
    dinov2_model=infer_backbone, n_slots=N_SLOT, cls_dim=CFG['cls_dim'],
    n_classes=CFG['num_classes'], slot_hidden=CFG['slot_hidden'],
    dropout=0.0, unfreeze_layers=CFG['unfreeze_layers'],
).to(DEVICE)

# 加载权重（处理 DataParallel 前缀 + EMA）
state_dict = ckpt['model']
first_key = next(iter(state_dict))
if first_key.startswith('module.'):
    state_dict = {k.replace('module.', '', 1): v for k, v in state_dict.items()}

# ★ 优先使用 EMA 权重
if ckpt.get('ema') and ckpt['ema'].get('shadow'):
    for name in state_dict:
        ema_key = name
        if ema_key in ckpt['ema']['shadow']:
            state_dict[name] = ckpt['ema']['shadow'][ema_key]
    print('  Using EMA weights for inference')

infer_model.load_state_dict(state_dict, strict=False)
infer_model.eval()
print(f'  Model loaded: epoch={ckpt.get("epoch")}, AUC={ckpt.get("auc", 0):.4f}')

# ============================================================
# Part A: Gold Validation (7-window TTA + 诊断池化)
# ============================================================

print('\n--- Gold Validation ---')
gold_cache = SLOT_CACHE  # reuse training cache
gold_mask_arr = SLOT_MASK
gold_val_uids = sorted(val_gold_uids)

# Filter to cached gold studies
cached_gold = [u for u in gold_val_uids if u in study_index]
print(f'Gold studies in cache: {len(cached_gold)}/{len(gold_val_uids)}')

# Build gold validation dataset with all 7 windows
N_WINDOWS = CFG['cache_slices'] - CFG['group_size'] + 1  # 7

gold_rows = []
for uid in cached_gold:
    ri = study_index[uid]
    slots_all = torch.from_numpy(gold_cache[ri].copy())  # [6, 9, H, W]
    m = torch.from_numpy(gold_mask_arr[ri].copy())        # [6]
    windows = torch.stack([slots_all[:, w:w+CFG['group_size']] for w in range(N_WINDOWS)], dim=0)
    gold_rows.append((windows, m, uid))

# 分批推理
gold_labels_map = gold_labels
gold_probs_list, gold_uids_list = [], []

@torch.no_grad()
def infer_gold_batch(windows_batch, mask_batch, model):
    """TTA + 诊断池化（jitter 视图平均 → per-target 窗口池化）"""
    B = windows_batch.shape[0]
    W = N_WINDOWS
    flat = windows_batch.reshape(B * W, *windows_batch.shape[2:]).to(DEVICE)  # B-major
    flat_mask = mask_batch.unsqueeze(1).expand(B, W, -1).reshape(B * W, -1).to(DEVICE)
    if CFG.get('tta_jitter', False):
        flat = torch.cat([flat, tta_jitter(flat)], dim=0)   # [2*B*W, ...] 原始块在前
        flat_mask = flat_mask.repeat(2, 1)
        n_orig = W
    else:
        n_orig = None
    logits = model(flat, flat_mask)  # [V*B*W, 12]
    logits_v = stack_views(logits, B, W, n_orig)  # [B, V*W, 12]
    return diagnostic_pool(logits_v.cpu(), n_orig=n_orig)  # [B, 12]

for start in range(0, len(gold_rows), 8):
    batch = gold_rows[start:start+8]
    windows_batch = torch.stack([r[0] for r in batch])
    mask_batch = torch.stack([r[1] for r in batch])
    uids = [r[2] for r in batch]

    probs = infer_gold_batch(windows_batch, mask_batch, infer_model)
    gold_probs_list.append(probs)
    gold_uids_list.extend(uids)

gold_probs_all = torch.cat(gold_probs_list).numpy()

# Build labels
gold_labels_arr = np.zeros((len(gold_uids_list), N_CLASSES))
for i, uid in enumerate(gold_uids_list):
    for j, c in enumerate(TARGET_COLUMNS):
        raw = gold_labels_map.loc[uid, c] if uid in gold_labels_map.index else np.nan
        gold_labels_arr[i, j] = float(raw) if not pd.isna(raw) else 0.0

# AUC
gold_aucs = {}
for i, c in enumerate(TARGET_COLUMNS):
    yt, yp = gold_labels_arr[:, i], gold_probs_all[:, i]
    # Only labeled studies (all should be labeled for gold)
    valid = yt >= 0
    yt, yp = yt[valid], yp[valid]
    n_pos = int(yt.sum())
    if n_pos > 0 and n_pos < len(yt):
        try: gold_aucs[c] = float(roc_auc_score(yt, yp))
        except Exception: gold_aucs[c] = float('nan')
    else: gold_aucs[c] = float('nan')

valid_aucs = [v for v in gold_aucs.values() if not math.isnan(v)]
gold_macro = float(np.mean(valid_aucs)) if valid_aucs else float('nan')

print(f'\nGold Validation ({len(gold_uids_list)} studies, '
      f'{N_WINDOWS}-window TTA{"+jitter" if CFG.get("tta_jitter", False) else ""} + diag pool):')
print(f'  {"Class":<20s} {"AUC":>7s} {"Pos":>5s}')
for i, c in enumerate(TARGET_COLUMNS):
    a = gold_aucs[c]
    auc_s = f'{a:.4f}' if not math.isnan(a) else '  N/A  '
    print(f'  {c:<20s} {auc_s:>7s} {int(gold_labels_arr[:, i].sum()):5d}')
print(f'  {"Macro AUC":<20s} {gold_macro:7.4f}')

# ============================================================
# Part B: Test Set Inference → submission.csv
# ============================================================

print('\n--- Test Set Inference ---')

# Load test metadata
test_df = pd.read_csv(comp_input / 'test.csv')
test_df['StudyInstanceUID'] = test_df['StudyInstanceUID'].astype(str)

# ---- Build test series metadata ----
# 优先使用 test_series.csv；如果不足，扫描 DICOM 目录构建元数据
test_dicom_root = comp_input / 'test_series'

def _find_dicom_files(series_dir):
    """列出目录中的 DICOM 文件（不依赖扩展名，竞赛 test 集 DICOM 无 .dcm 后缀）。"""
    all_files = sorted([f for f in series_dir.iterdir() if f.is_file()])
    # 优先 .dcm 后缀；若无，取所有文件（跳过隐藏文件）
    dcm = [f for f in all_files if f.suffix == '.dcm']
    return dcm if dcm else [f for f in all_files if not f.name.startswith('.')]


def _scan_test_dicoms(dicom_root):
    """扫描测试集 DICOM 目录，从 header 推断 plane / fluid / fatsat。"""
    rows = []
    root = Path(dicom_root)
    if not root.exists():
        return rows
    for study_dir in sorted(root.iterdir()):
        if not study_dir.is_dir():
            continue
        study_uid = study_dir.name
        for series_dir in sorted(study_dir.iterdir()):
            if not series_dir.is_dir():
                continue
            series_uid = series_dir.name
            dcm_files = _find_dicom_files(series_dir)
            if not dcm_files:
                continue
            try:
                ds = pydicom.dcmread(str(dcm_files[0]), stop_before_pixels=True, force=True)

                # ★ Anatomical Plane (from ImageOrientationPatient)
                iop = getattr(ds, 'ImageOrientationPatient', None)
                plane = 'Axial'  # default
                if iop is not None and len(iop) >= 6:
                    try:
                        row_cos = np.array([float(iop[0]), float(iop[1]), float(iop[2])])
                        col_cos = np.array([float(iop[3]), float(iop[4]), float(iop[5])])
                        normal = np.cross(row_cos, col_cos)
                        dominant = int(np.argmax(np.abs(normal)))
                        plane = {0: 'Sagittal', 1: 'Coronal', 2: 'Axial'}[dominant]
                    except Exception:
                        pass

                # ★ Fat Suppression (from ScanOptions / SeriesDescription)
                desc = str(getattr(ds, 'SeriesDescription', '')).lower()
                seq_name = str(getattr(ds, 'SequenceName', '')).lower()
                scan_opts = str(getattr(ds, 'ScanOptions', '')).upper()

                fs_kw = ['fs', 'fatsat', 'fat sat', 'stir', 'spair', 'spir', 'we',
                         'water excit', 'tirm', 'fatsup']
                has_fs = any(kw in desc for kw in fs_kw)
                has_fs = has_fs or any(kw in scan_opts for kw in ['FS', 'FATSAT', 'SPAIR', 'SPIR'])

                # ★ Fluid Sensitive (T2 / PD weighted)
                t1_kw = ['t1', 't1w']
                is_t1 = any(kw in desc or kw in seq_name for kw in t1_kw)
                is_t2 = any(kw in desc or kw in seq_name for kw in ['t2', 't2w'])
                is_pd = any(kw in desc for kw in ['pd', 'pdw', 'proton', 'dp', 'dens'])
                has_fluid = (is_t2 or is_pd) and not is_t1

                rows.append({
                    'StudyInstanceUID': study_uid,
                    'SeriesInstanceUID': series_uid,
                    'Anatomical_Plane': plane,
                    'Fluid_Sensitive': 1 if has_fluid else 0,
                    'Fat_Suppression': 1 if has_fs else 0,
                })
            except Exception:
                continue
    return rows


# ★ 重写逻辑：CSV 结果不会被 DICOM scan 失败覆盖
test_slot_map = {}
test_series_path = comp_input / 'test_series.csv'

if test_series_path.exists():
    test_series = pd.read_csv(test_series_path)
    test_series['StudyInstanceUID'] = test_series['StudyInstanceUID'].astype(str)
    test_series['SeriesInstanceUID'] = test_series['SeriesInstanceUID'].astype(str)
    print(f'test_series.csv: {len(test_series)} series, '
          f'{test_series["StudyInstanceUID"].nunique()} studies')

    # 先走 CSV 路径
    test_slot_map, _ = build_study_slot_map(test_series, test_dicom_root)
    csv_studies = len(test_slot_map)

    # ★ 如果 CSV 覆盖不足（< 50% test studies），扫描 DICOM 补充
    if csv_studies < max(10, len(test_df) * 0.5):
        print(f'CSV coverage ({csv_studies}/{len(test_df)}) insufficient, '
              f'scanning DICOM headers...')
        dicom_rows = _scan_test_dicoms(test_dicom_root)
        if dicom_rows:
            test_series = pd.DataFrame(dicom_rows)
            test_slot_map, _ = build_study_slot_map(test_series, test_dicom_root)
            print(f'DICOM scan: {len(test_series)} series, '
                  f'{test_series["StudyInstanceUID"].nunique()} studies → '
                  f'{len(test_slot_map)} studies matched')
        else:
            print(f'DICOM scan returned 0 rows, keeping CSV results ({csv_studies} studies)')
    # else: CSV 覆盖率够了，直接用
else:
    print('test_series.csv not found, scanning DICOM headers...')
    dicom_rows = _scan_test_dicoms(test_dicom_root)
    if dicom_rows:
        test_series = pd.DataFrame(dicom_rows)
        test_slot_map, _ = build_study_slot_map(test_series, test_dicom_root)
        print(f'DICOM scan: {len(test_series)} series, '
              f'{len(test_slot_map)} studies matched')

test_studies = sorted(test_slot_map.keys())
print(f'Test studies with slot match: {len(test_studies)}/{len(test_df)}')

# ★ 释放训练缓存，为测试缓存腾出内存
del SLOT_CACHE, SLOT_MASK
gc.collect()
print(f'Freed train cache for test set (GPU: {torch.cuda.memory_allocated()/1024**3:.1f} GB)')

# Build test cache + inference (if DICOMs available)
if len(test_studies) > 0:
    n_test = len(test_studies)
    TEST_CACHE = np.zeros((n_test, N_SLOT, CFG['cache_slices'], CFG['image_size'], CFG['image_size']), dtype=np.uint8)
    TEST_MASK = np.zeros((n_test, N_SLOT), dtype=np.float32)
    test_study_idx = {}

    t0 = time.time()
    jobs = []
    for row_idx, study_uid in enumerate(test_studies):
        test_study_idx[study_uid] = row_idx
        study_slots = test_slot_map[study_uid]
        for slot_idx, (slot_name, plane, fluid, fatsat) in enumerate(SLOTS):
            slot_info = study_slots.get(slot_name)
            if slot_info is not None:
                jobs.append((row_idx, slot_idx, slot_name, plane, slot_info, None))

    print(f'Decoding {len(jobs)} test slot-series...')
    completed, failed = 0, 0
    with ThreadPoolExecutor(max_workers=CFG['pix_threads']) as pool:
        for row_idx, slot_idx, result in pool.map(_read_slot_job, jobs):
            completed += 1
            if result is not None:
                TEST_CACHE[row_idx, slot_idx] = result
                TEST_MASK[row_idx, slot_idx] = 1.0
            else:
                failed += 1
            if completed % 1000 == 0:
                print(f'  [{completed}/{len(jobs)}] {time.time()-t0:.0f}s', flush=True)

    print(f'Test cache: {n_test} studies in {time.time()-t0:.0f}s ({failed} failed)')

    # TTA inference on test set
    print('Running TTA inference on test set...')
    test_probs = np.zeros((n_test, N_CLASSES), dtype=np.float32)

    @torch.no_grad()
    def infer_test_batch(indices, model):
        windows_list, masks_list, empty_mask = [], [], []
        for idx in indices:
            windows_list.append(torch.stack([
                torch.from_numpy(TEST_CACHE[idx, :, w:w+CFG['group_size']].copy())
                for w in range(N_WINDOWS)
            ], dim=0))
            masks_list.append(torch.from_numpy(TEST_MASK[idx].copy()))
            empty_mask.append(TEST_MASK[idx].sum() == 0)  # Track studies with no slots

        windows_batch = torch.stack(windows_list)
        mask_batch = torch.stack(masks_list)

        B, W = windows_batch.shape[0], N_WINDOWS
        flat = windows_batch.reshape(B * W, *windows_batch.shape[2:]).to(DEVICE)  # B-major
        flat_mask = mask_batch.unsqueeze(1).expand(B, W, -1).reshape(B * W, -1).to(DEVICE)
        if CFG.get('tta_jitter', False):
            flat = torch.cat([flat, tta_jitter(flat)], dim=0)   # [2*B*W, ...] 原始块在前
            flat_mask = flat_mask.repeat(2, 1)
            n_orig = W
        else:
            n_orig = None
        logits = model(flat, flat_mask)
        probs = diagnostic_pool(
            stack_views(logits, B, W, n_orig).cpu(), n_orig=n_orig)  # [B, C]

        # Studies with no slots → fill 0.5
        for i, is_empty in enumerate(empty_mask):
            if is_empty:
                probs[i] = 0.5
        return probs

    t1 = time.time()
    for start in range(0, n_test, 8):
        idx = list(range(start, min(start + 8, n_test)))
        test_probs[idx] = infer_test_batch(idx, infer_model).numpy()
        if start % 200 == 0:
            print(f'  [{start}/{n_test}] {time.time()-t1:.0f}s', flush=True)

    print(f'Test inference done in {time.time()-t1:.0f}s')

    # Build submission rows from inference results
    submission_rows = []
    for row_idx, study_uid in enumerate(test_studies):
        row = {'StudyInstanceUID': study_uid}
        for j, c in enumerate(TARGET_COLUMNS):
            row[c] = float(test_probs[row_idx, j])
        submission_rows.append(row)
else:
    print('No test DICOMs found — filling all studies with 0.5')
    submission_rows = []

submission_df = pd.DataFrame(submission_rows)

# Ensure all test studies are present (fill missing with 0.5)
full_submission = test_df[['StudyInstanceUID']].merge(
    submission_df, on='StudyInstanceUID', how='left')
for c in TARGET_COLUMNS:
    full_submission[c] = full_submission[c].fillna(0.5)

submission_path = output_dir / 'submission.csv'
full_submission.to_csv(submission_path, index=False)
print(f'\nSubmission saved: {submission_path}')
print(f'  Studies: {len(full_submission)} (expected: {len(test_df)})')
print(f'  Mean prob: {full_submission[TARGET_COLUMNS].values.mean():.4f}')

# Quick stats
for c in TARGET_COLUMNS:
    vals = full_submission[c].values
    print(f'  {c:<20s}: mean={vals.mean():.4f}, std={vals.std():.4f}, '
          f'>0.5={np.mean(vals>0.5):.1%}')

# ---- Save gold validation results ----
gold_rows_out = []
for i, uid in enumerate(gold_uids_list):
    row = {'StudyInstanceUID': uid}
    for j, c in enumerate(TARGET_COLUMNS):
        row[f'true_{c}'] = int(gold_labels_arr[i, j])
        row[f'prob_{c}'] = float(gold_probs_all[i, j])
    gold_rows_out.append(row)
pd.DataFrame(gold_rows_out).to_csv(
    output_dir / f'gold_validation_predictions_{SEED_TAG}.csv', index=False)

auc_rows = [{'class': c, 'auc': gold_aucs[c], 'n_pos': int(gold_labels_arr[:, i].sum())}
            for i, c in enumerate(TARGET_COLUMNS)]
pd.DataFrame(auc_rows + [{'class': 'macro_avg', 'auc': gold_macro, 'n_pos': 0}]
            ).to_csv(output_dir / f'gold_validation_auc_{SEED_TAG}.csv', index=False)

print(f'\nDone!')
print(f'  Gold AUC: {gold_macro:.4f}')
print(f'  Submission: {submission_path}')
print(f'  Ready to submit to Kaggle!')

_phase3d_final = dict(_preflight, gold_macro_auc=gold_macro, best_epoch=best_epoch)
(output_dir / 'phase3d_manifest.json').write_text(json.dumps(_phase3d_final, indent=2))
